# Stream a Changefeed to Databricks

> ⚠️ **SYNC NOTE**: This notebook is synchronized with `STREAM_CHANGEFEED_TO_DATABRICKS.md`. When updating one file, please update the other to maintain consistency.
> 
> 📚 **FOR PRODUCTION DEPLOYMENTS**: See the comprehensive Azure-specific guide at `sources/cockroachdb/docs/stream-changefeed-to-databricks-azure.md`, which includes schema file requirements, SDP configuration, Community Edition compatibility, and TB-scale deployment guidance.

**For submission to:** CockroachDB Documentation  
**Author:** Lakeflow Community Connectors  
**Date:** 2025-12-23

---

While CockroachDB is an excellent system of record, it also needs to coexist with other systems. For example, you might want to keep your data mirrored in data lakes, analytics engines, or machine learning pipelines.

This page demonstrates how to use a changefeed to stream row-level changes to Databricks, a unified analytics platform built on Apache Spark and Delta Lake.

> **Note:** Databricks Autoloader automatically handles schema inference, evolution, and deduplication, making it significantly simpler than traditional ETL pipelines. This tutorial shows how to stream data to Azure Blob Storage with Databricks Autoloader automatically ingesting changes into Delta Lake tables.

## Before you begin

Before you begin, make sure you have:

* Admin access to a CockroachDB Cloud account
* Write access to an Azure Blob Storage account  
  **Note:** This tutorial uses Azure Blob Storage for cloud storage. CockroachDB also supports AWS S3 and Google Cloud Storage. The Databricks Autoloader pattern works identically across all three cloud providers.
* Read and write access to a Databricks workspace (Standard or Premium tier)
* The `CHANGEFEED` privilege in order to create and manage changefeed jobs. Refer to [Required privileges](https://www.cockroachlabs.com/docs/stable/create-changefeed#required-privileges) for more details.

## Architecture Overview

This tutorial creates a streaming CDC pipeline with:

```
CockroachDB → Azure Blob Storage → Databricks Autoloader → Delta Lake
```

**Key advantages over other approaches:**

* **No middleware required:** No SQS queues, Snowpipes, or custom ETL
* **Automatic schema inference:** Databricks detects schema changes automatically
* **Built-in deduplication:** Delta Lake MERGE handles updates and duplicates
* **Support for deletes:** Full CDC operations (INSERT, UPDATE, DELETE)
* **Cost-effective:** Only storage costs, no additional compute for ingestion
* **Multi-format support:** Works with both JSON and Parquet

---

## Steps 1-8: CockroachDB Configuration

The following steps should be executed in your CockroachDB SQL shell. This notebook focuses on the Databricks configuration (Steps 9-10).

In [ ]:
# Reference: CockroachDB Setup Commands
# Execute these in your CockroachDB SQL shell before proceeding

cockroachdb_commands = """
-- Step 3: Enable rangefeeds
SET CLUSTER SETTING kv.rangefeed.enabled = true;

-- Step 4: Create database and schema
CREATE DATABASE ecommerce;
SET DATABASE = ecommerce;
CREATE SCHEMA IF NOT EXISTS public;

-- Step 5: Create orders table
CREATE TABLE public.orders (
    order_id    UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    customer_id INT NOT NULL,
    product_id  INT NOT NULL,
    quantity    INT NOT NULL,
    total       DECIMAL(10,2) NOT NULL,
    status      STRING NOT NULL,
    created_at  TIMESTAMPTZ DEFAULT now(),
    updated_at  TIMESTAMPTZ DEFAULT now()
);

-- Insert sample data
INSERT INTO public.orders (customer_id, product_id, quantity, total, status)
VALUES 
    (1001, 5001, 2, 49.98, 'pending'),
    (1002, 5002, 1, 29.99, 'pending'),
    (1003, 5003, 3, 89.97, 'processing');

-- Step 7: Create changefeed (Parquet format - recommended)
CREATE CHANGEFEED FOR TABLE ecommerce.public.orders
INTO 'azure://changefeed-events/parquet/ecommerce/public/?AZURE_ACCOUNT_NAME={your-storage-account-name}&AZURE_ACCOUNT_KEY={your-storage-account-key}'
WITH 
    format = 'parquet',
    compression = 'gzip',
    updated,
    resolved = '10s',
    initial_scan = 'yes';

-- Step 8: Insert test data
INSERT INTO public.orders (customer_id, product_id, quantity, total, status)
VALUES (1004, 5004, 1, 19.99, 'pending');

UPDATE public.orders 
SET status = 'shipped', updated_at = now() 
WHERE customer_id = 1001;

DELETE FROM public.orders WHERE customer_id = 1002;
"""

print("="*80)
print("COCKROACHDB SETUP COMMANDS")
print("="*80)
print(cockroachdb_commands)
print("="*80)
print("✅ After executing these commands in CockroachDB, proceed with the cells below")

## Step 9: Configure Databricks Autoloader

### Configure Azure Storage Credentials

Replace the placeholders with your actual Azure storage account details:

In [ ]:
# Configure Azure storage access
storage_account_name = "your-storage-account-name"  # ← Replace with your storage account
storage_account_key = "your-storage-account-key"      # ← Replace with your access key

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net",
    storage_account_key
)

print("✅ Azure storage credentials configured")
print(f"📦 Storage Account: {storage_account_name}")

### Configure Autoloader for Parquet CDC Files

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ============================================================================
# CONFIGURATION - Update these values
# ============================================================================
storage_account_name = "your-storage-account-name"  # ← Replace
target_catalog = "ecommerce_catalog"                 # ← Replace
target_schema = "public"                             # ← Replace

source_path = f"wasbs://changefeed-events@{storage_account_name}.blob.core.windows.net/parquet/ecommerce/public/orders/"
checkpoint_path = "/checkpoints/ecommerce/public/orders/parquet"
target_table = f"{target_catalog}.{target_schema}.orders"

print("="*80)
print("DATABRICKS AUTOLOADER - PARQUET CDC INGESTION")
print("="*80)
print(f"📂 Source: {source_path}")
print(f"📊 Target: {target_table}")
print(f"💾 Checkpoint: {checkpoint_path}")
print("="*80)

# ============================================================================
# STEP 1: Read streaming data with Autoloader
# ============================================================================
print("\n🔄 Step 1: Configuring Autoloader...")
raw_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)
print("✅ Autoloader configured")

# ============================================================================
# STEP 2: Process CDC events
# ============================================================================
print("\n🔄 Step 2: Processing CDC events...")
processed_stream = raw_stream.select(
    "*",
    F.when(F.col("__crdb__event_type") == "d", "DELETE")
     .otherwise("UPSERT")
     .alias("_cdc_operation"),
    F.col("__crdb__updated").alias("_cdc_updated")
)

# Deduplicate by primary key (handles split_column_families)
window = Window.partitionBy("order_id").orderBy(F.col("_cdc_updated").desc())
deduped_stream = processed_stream \
    .withColumn("row_num", F.row_number().over(window)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num", "__crdb__event_type", "__crdb__updated")
print("✅ CDC processing configured")

# ============================================================================
# STEP 3: Define merge logic
# ============================================================================
print("\n🔄 Step 3: Defining merge logic...")

def merge_to_delta(micro_batch_df, epoch_id):
    """Merge CDC events into Delta table with upsert/delete logic."""
    
    # Create table if it doesn't exist
    if not spark.catalog.tableExists(target_table):
        micro_batch_df.write.format("delta").saveAsTable(target_table)
        print(f"  ✅ Created new table: {target_table}")
        return
    
    # Merge into existing table
    delta_table = DeltaTable.forName(spark, target_table)
    
    delta_table.alias("target").merge(
        micro_batch_df.alias("source"),
        "target.order_id = source.order_id"
    ).whenMatchedUpdate(
        condition="source._cdc_operation = 'UPSERT' AND source._cdc_updated > target._cdc_updated",
        set={
            "customer_id": "source.customer_id",
            "product_id": "source.product_id",
            "quantity": "source.quantity",
            "total": "source.total",
            "status": "source.status",
            "created_at": "source.created_at",
            "updated_at": "source.updated_at",
            "_cdc_updated": "source._cdc_updated"
        }
    ).whenMatchedDelete(
        condition="source._cdc_operation = 'DELETE'"
    ).whenNotMatchedInsert(
        condition="source._cdc_operation = 'UPSERT'",
        values={
            "order_id": "source.order_id",
            "customer_id": "source.customer_id",
            "product_id": "source.product_id",
            "quantity": "source.quantity",
            "total": "source.total",
            "status": "source.status",
            "created_at": "source.created_at",
            "updated_at": "source.updated_at",
            "_cdc_updated": "source._cdc_updated"
        }
    ).execute()
    print(f"  ✅ Merged batch {epoch_id}")

print("✅ Merge function defined")

# ============================================================================
# STEP 4: Start streaming
# ============================================================================
print("\n🔄 Step 4: Starting streaming query...")
query = (deduped_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint_path}/data")
    .foreachBatch(merge_to_delta)
    .trigger(availableNow=True)  # Process all available data then stop
    .start()
)

print("✅ Streaming query started")
print("⏳ Processing data...")

# Wait for completion
query.awaitTermination()

print("\n" + "="*80)
print("✅ STREAMING QUERY COMPLETED SUCCESSFULLY")
print("="*80)
print(f"📊 Query your data: SELECT * FROM {target_table}")

# Store target_table for next cell
dbutils.widgets.text("target_table", target_table, "Target Table")

In [ ]:
-- Query the ingested orders
SELECT * FROM ecommerce_catalog.public.orders 
ORDER BY updated_at DESC 
LIMIT 100;

## ✅ Congratulations!

Your CockroachDB changefeed is now streaming to Databricks with:
- ✅ Automatic schema inference
- ✅ Built-in deduplication  
- ✅ Full CDC support (INSERT, UPDATE, DELETE)
- ✅ Delta Lake ACID transactions

## Next Steps

1. **Monitor your changefeed** in CockroachDB: `SHOW CHANGEFEED JOBS;`
2. **Optimize performance**: Run `OPTIMIZE ecommerce_catalog.public.orders ZORDER BY (order_id)`
3. **Enable time travel**: Query historical data with `SELECT * FROM table_name TIMESTAMP AS OF '2025-01-26'`
4. **Set up alerts**: Configure Databricks SQL alerts for data freshness
5. **Extend to more tables**: Create additional changefeeds for other tables

## Resources

- 📚 [Full Documentation](./STREAM_CHANGEFEED_TO_DATABRICKS.md) - Complete guide with JSON format examples
- 🔗 [Databricks Autoloader](https://docs.databricks.com/ingestion/auto-loader/index.html)
- 🔗 [Delta Lake MERGE](https://docs.databricks.com/delta/merge.html)
- 🔗 [CockroachDB Changefeeds](https://www.cockroachlabs.com/docs/stable/create-changefeed)

---

> ⚠️ **SYNC NOTE**: This notebook is synchronized with `STREAM_CHANGEFEED_TO_DATABRICKS.md`. When updating one file, please update the other to maintain consistency.

pwd